# Run 3 - Word2Vec + SVM vs CamemBERT + SVM


In [1]:
!pip install -q gensim transformers torch scikit-learn pandas sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.8 MB/s eta 0:00:00


In [2]:
import gc
import re
import numpy as np
import pandas as pd
import torch
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from transformers import AutoTokenizer, AutoModel
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report
from google.colab import files

print("Imports OK")
print(f"GPU disponible : {torch.cuda.is_available()}")

Imports OK
GPU disponible : True


In [4]:

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train : {len(train)} recettes")
print(f"Test  : {len(test)} recettes")

assert len(train) == 12473, f"ERREUR train incomplet : {len(train)} lignes"
assert len(test)  == 1388,  f"ERREUR test incomplet : {len(test)} lignes"

y_train = train["type"]
y_test  = test["type"]

Train : 12473 recettes
Test  : 1388 recettes


In [5]:
# Preprocessing - texte complet pour les deux methodes
def preprocess(text):
    text = re.sub(r"[^\w\s]", " ", str(text))
    text = re.sub(r"\d+", "", text)
    return text.lower()

def build_text(df, columns):
    return df[columns].fillna("").astype(str).agg(" ".join, axis=1)

# titre + ingredients + recette pour les deux
x_train_text = build_text(train, ["titre", "ingredients", "recette"]).apply(preprocess)
x_test_text  = build_text(test,  ["titre", "ingredients", "recette"]).apply(preprocess)

print("Preprocessing OK")

Preprocessing OK


## Run 3a - Word2Vec + SVM

In [7]:
sentences = [simple_preprocess(t) for t in x_train_text]

w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    seed=42
)
print(f"Word2Vec entraine - vocab : {len(w2v.wv)} mots")

Word2Vec entraine - vocab : 11866 mots


In [8]:
def doc_vector(text, model):
    tokens = simple_preprocess(str(text))
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0).astype(np.float32) if vecs else np.zeros(model.vector_size, dtype=np.float32)

print("Vectorisation train...")
x_train_w2v = np.array([doc_vector(t, w2v) for t in x_train_text], dtype=np.float32)
print("Vectorisation test...")
x_test_w2v  = np.array([doc_vector(t, w2v) for t in x_test_text],  dtype=np.float32)

print(f"Shape train : {x_train_w2v.shape}, test : {x_test_w2v.shape}")

Vectorisation train...
Vectorisation test...
Shape train : (12473, 100), test : (1388, 100)


In [9]:
clf_w2v = SVC(kernel="linear", C=1.0, random_state=42)
clf_w2v.fit(x_train_w2v, y_train)
y_pred_w2v = clf_w2v.predict(x_test_w2v)

f1_w2v = f1_score(y_test, y_pred_w2v, average="macro")

print("=== Run 3a - Word2Vec + SVM ===")
print(classification_report(y_test, y_pred_w2v))
print(f"F1 macro Word2Vec : {f1_w2v:.3f}")

pd.Series(y_pred_w2v).to_csv("predictions_run3a_word2vec.csv", index=False)

=== Run 3a - Word2Vec + SVM ===
                precision    recall  f1-score   support

       Dessert       0.98      0.99      0.99       407
        Entrée       0.75      0.66      0.70       337
Plat principal       0.83      0.89      0.86       644

      accuracy                           0.86      1388
     macro avg       0.86      0.84      0.85      1388
  weighted avg       0.86      0.86      0.86      1388

F1 macro Word2Vec : 0.848


## Run 3b - CamemBERT + SVM (texte complet)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

tokenizer  = AutoTokenizer.from_pretrained("camembert-base")
model_bert = AutoModel.from_pretrained("camembert-base").to(device)
model_bert.eval()
print("CamemBERT charge")

Device : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CamemBERT charge


In [11]:
# max_length=256 pour accommoder titre + ingredients + recette
def encode_texts(texts, tokenizer, model, device, max_length=256):
    all_embeddings = []
    texts = list(texts)
    batch_size = 32 if torch.cuda.is_available() else 8

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(**inputs)

        emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
        all_embeddings.append(emb)

        del inputs, out, emb
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size, len(texts))}/{len(texts)}")

    return np.vstack(all_embeddings).astype(np.float32)

print("Extraction embeddings train...")
x_train_bert = encode_texts(x_train_text, tokenizer, model_bert, device)
print("Extraction embeddings test...")
x_test_bert  = encode_texts(x_test_text,  tokenizer, model_bert, device)

print(f"Shape train : {x_train_bert.shape}, test : {x_test_bert.shape}")

Extraction embeddings train...
  32/12473
  672/12473
  1312/12473
  1952/12473
  2592/12473
  3232/12473
  3872/12473
  4512/12473
  5152/12473
  5792/12473
  6432/12473
  7072/12473
  7712/12473
  8352/12473
  8992/12473
  9632/12473
  10272/12473
  10912/12473
  11552/12473
  12192/12473
Extraction embeddings test...
  32/1388
  672/1388
  1312/1388
Shape train : (12473, 768), test : (1388, 768)


In [12]:
clf_bert = SVC(kernel="linear", C=1.0, random_state=42)
clf_bert.fit(x_train_bert, y_train)
y_pred_bert = clf_bert.predict(x_test_bert)

f1_bert = f1_score(y_test, y_pred_bert, average="macro")

print("=== Run 3b - CamemBERT + SVM ===")
print(classification_report(y_test, y_pred_bert))
print(f"F1 macro CamemBERT : {f1_bert:.3f}")

pd.Series(y_pred_bert).to_csv("predictions_run3b_camembert.csv", index=False)

=== Run 3b - CamemBERT + SVM ===
                precision    recall  f1-score   support

       Dessert       0.97      1.00      0.98       407
        Entrée       0.75      0.59      0.66       337
Plat principal       0.82      0.89      0.85       644

      accuracy                           0.85      1388
     macro avg       0.85      0.83      0.83      1388
  weighted avg       0.85      0.85      0.85      1388

F1 macro CamemBERT : 0.833


In [14]:
print("=" * 55)
print("RECAP - Run 3")
print("=" * 55)
print(f"Word2Vec  (statique)   F1 macro = {f1_w2v:.3f}")
print(f"CamemBERT (dynamique)  F1 macro = {f1_bert:.3f}")

files.download("predictions_run3a_word2vec.csv")
files.download("predictions_run3b_camembert.csv")

RECAP - Run 3
Word2Vec  (statique)   F1 macro = 0.848
CamemBERT (dynamique)  F1 macro = 0.833


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>